In [ ]:
# Import required libraries
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
# Setup Driver
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

In [ ]:
# Scroll Until All Cars Load 
def load_all_cars(driver, brand, city, step=800, pause=2, max_retries=5):
    url = f"https://www.cars24.com/buy-used-{brand}-cars-{city}/"
    driver.get(url)

    # Wait until first car is visible
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.styles_carCardWrapper__sXLIp"))
    )

    retries = 0
    last_count = 0

    while True:
        # Scroll step by step like a user
        driver.execute_script(f"window.scrollBy(0, {step});")
        time.sleep(pause)

        cars = driver.find_elements(By.CSS_SELECTOR, "a.styles_carCardWrapper__sXLIp")
        new_count = len(cars)

        if new_count == last_count:  # no new cars loaded
            retries += 1
        else:
            retries = 0

        if retries >= max_retries:
            break

        last_count = new_count

    car_links = [c.get_attribute("href") for c in cars]
    print(f"[{brand}-{city}] Cars found: {len(car_links)}")
    return car_links

In [4]:
# Extract Car Details
def extract_car_details(driver, link, city):
    driver.get(link)
    time.sleep(3)

    car_data = {"location": city}

    # Title => Year, Brand, Model
    try:
        title = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        ).text
        parts = title.split()
        car_data["year"] = parts[0]
        car_data["brand"] = parts[1]
        car_data["model"] = " ".join(parts[2:])
    except:
        car_data["year"] = car_data["brand"] = car_data["model"] = None

    # Price
    try:
        car_data["price"] = driver.find_element(
            By.CSS_SELECTOR, "p[class='sc-braxZu hhzsvw']"
        ).text
    except:
        car_data["price"] = None

    # Specs => KM, Owners, Transmission, Fuel
    try:
        specs = driver.find_elements(By.CSS_SELECTOR, "p.kvfdZL")
        specs_list = [s.text for s in specs if s.text.strip()]

        if len(specs_list) >= 4:
            car_data["km_driven"] = specs_list[0]
            car_data["no_owners"] = specs_list[1]
            car_data["transmission"] = specs_list[2]
            car_data["fuel_type"] = specs_list[3]
    except:
        car_data["km_driven"] = car_data["no_owners"] = car_data["transmission"] = car_data["fuel_type"] = None

    return car_data


In [5]:
# Main Scraping Function
def scrape_cars24(brand, city):
    driver = setup_driver()
    car_links = load_all_cars(driver, brand, city)

    all_data = []
    for link in car_links:
        data = extract_car_details(driver, link, city)
        all_data.append(data)

    driver.quit()
    df = pd.DataFrame(all_data)
    return df

In [ ]:
# Main Execution
if __name__ == "__main__":
    brand = "honda"
    cities = ["ahmedabad", "hyderabad", "mumbai"]

    for city in cities:
        df = scrape_cars24(brand, city)
        print(df.head())
        df.to_csv(f"{brand}_{city}_cars.csv", index=False)
        print(f"Data saved: {brand}_{city}_cars.csv")

[honda-ahmedabad] Cars found: 115
    location  year  brand                     model        price  km_driven  \
0  ahmedabad  2024  Honda              ELEVATE V MT  ₹10.94 lakh   8,928 km   
1  ahmedabad  2017  Honda    WR-V 1.2L I-VTEC VX MT   ₹4.52 lakh  61,762 km   
2  ahmedabad  2023  Honda   Amaze 1.2L I-VTEC S CVT   ₹6.46 lakh  27,812 km   
3  ahmedabad  2021  Honda  Amaze 1.2L I-VTEC VX CVT   ₹7.06 lakh  30,897 km   
4  ahmedabad  2014  Honda       Amaze 1.2L I-VTEC S    ₹3.2 lakh  60,052 km   

   no_owners transmission fuel_type  
0  1st owner       Manual    Petrol  
1  1st owner       Manual    Petrol  
2  1st owner    Automatic    Petrol  
3  1st owner    Automatic    Petrol  
4  1st owner       Manual    Petrol  
Data saved: honda_ahmedabad_cars.csv
[honda-hyderabad] Cars found: 79
    location  year  brand                    model       price  km_driven  \
0  hyderabad  2017  Honda    Jazz 1.2L I-VTEC V AT  ₹5.15 lakh  66,557 km   
1  hyderabad  2017  Honda    Jazz 1.2L 

In [8]:
pd.read_csv('honda_ahmedabad_cars.csv')

,location,year,brand,model,price,km_driven,no_owners,transmission,fuel_type
0,ahmedabad,2024,Honda,ELEVATE V MT,₹10.94 lakh,"8,928 km",1st owner,Manual,Petrol
1,ahmedabad,2017,Honda,WR-V 1.2L I-VTEC VX MT,₹4.52 lakh,"61,762 km",1st owner,Manual,Petrol
2,ahmedabad,2023,Honda,Amaze 1.2L I-VTEC S CVT,₹6.46 lakh,"27,812 km",1st owner,Automatic,Petrol
3,ahmedabad,2021,Honda,Amaze 1.2L I-VTEC VX CVT,₹7.06 lakh,"30,897 km",1st owner,Automatic,Petrol
4,ahmedabad,2014,Honda,Amaze 1.2L I-VTEC S,₹3.2 lakh,"60,052 km",1st owner,Manual,Petrol
...,...,...,...,...,...,...,...,...,...
110,ahmedabad,2015,Honda,Mobilio 1.5L I-DTEC S,₹3.89 lakh,"2,32,382 km",1st owner,Manual,Diesel
111,ahmedabad,2022,Honda,Jazz 1.2L I-VTEC V CVT,₹6 lakh,"52,275 km",1st owner,Automatic,Petrol
112,ahmedabad,2018,Honda,City 1.5L I-VTEC SV,₹5.28 lakh,"58,929 km",2nd owner,Manual,Petrol
113,ahmedabad,2019,Honda,Amaze 1.2L I-VTEC S,₹4.65 lakh,"77,907 km",2nd owner,Manual,CNG


In [7]:
pd.read_csv('honda_hyderabad_cars.csv')

,location,year,brand,model,price,km_driven,no_owners,transmission,fuel_type
0,hyderabad,2017,Honda,Jazz 1.2L I-VTEC V AT,₹5.15 lakh,"66,557 km",1st owner,Automatic,Petrol
1,hyderabad,2017,Honda,Jazz 1.2L I-VTEC V AT,₹5.38 lakh,"69,710 km",1st owner,Automatic,Petrol
2,hyderabad,2013,Honda,Brio V MT,₹2.56 lakh,"47,006 km",2nd owner,Manual,Petrol
3,hyderabad,2013,Honda,Amaze 1.2L I-VTEC VX,₹2.67 lakh,"60,414 km",3rd owner,Manual,CNG
4,hyderabad,2015,Honda,City 1.5L I-VTEC SV CVT,₹4.63 lakh,"77,391 km",1st owner,Automatic,Petrol
...,...,...,...,...,...,...,...,...,...
74,hyderabad,2020,Honda,City 1.5L I-VTEC ZX,₹11 lakh,"35,690 km",1st owner,Manual,Petrol
75,hyderabad,2020,Honda,City 1.5L I-VTEC ZX CVT,₹10.5 lakh,"64,770 km",1st owner,Automatic,Petrol
76,hyderabad,2022,Honda,City 1.5L I-VTEC ZX,₹11 lakh,"29,282 km",1st owner,Manual,Petrol
77,hyderabad,2023,Honda,ELEVATE ZX CVT,₹15.5 lakh,"29,771 km",1st owner,Automatic,Petrol


In [9]:
pd.read_csv('honda_mumbai_cars.csv')

,location,year,brand,model,price,km_driven,no_owners,transmission,fuel_type
0,mumbai,2018,Honda,BR-V 1.5L I-VTEC S,₹6.57 lakh,"29,807 km",1st owner,Manual,Petrol
1,mumbai,2015,Honda,Jazz 1.2L I-VTEC V AT,₹3.78 lakh,"76,301 km",1st owner,Automatic,Petrol
2,mumbai,2015,Honda,City 1.5L I-VTEC VX CVT,₹4.51 lakh,"1,01,436 km",2nd owner,Automatic,Petrol
3,mumbai,2017,Honda,WR-V 1.2L I-VTEC VX MT,₹4.79 lakh,"33,658 km",1st owner,Manual,Petrol
4,mumbai,2015,Honda,Jazz 1.2L I-VTEC VX,₹3.64 lakh,"20,247 km",1st owner,Manual,Petrol
...,...,...,...,...,...,...,...,...,...
140,mumbai,2015,Honda,City 1.5L I-VTEC V MT,₹3.8 lakh,"1,19,662 km",1st owner,Manual,Petrol
141,mumbai,2021,Honda,Amaze 1.2L I-VTEC S,₹5.41 lakh,"21,915 km",1st owner,Manual,Petrol
142,mumbai,2013,Honda,Amaze 1.2L I-VTEC VX AT,₹2.98 lakh,"19,385 km",1st owner,Automatic,Petrol
143,mumbai,2018,Honda,Jazz 1.2L I-VTEC VX,₹4.6 lakh,"29,540 km",1st owner,Manual,Petrol


In [89]:
# merging and cleaning csv into one csv file

In [1]:
#import required libraries
import pandas as pd
import re

In [91]:
# merging csv files
df1 = pd.read_csv("honda_hyderabad_cars.csv")
df2 = pd.read_csv("honda_ahmedabad_cars.csv")
df3 = pd.read_csv("honda_mumbai_cars.csv")

# Stack all rows into one DataFrame
merged_df = pd.concat([df1, df2, df3], ignore_index=True)

merged_df.to_csv("uncleaned_honda_cars.csv", index=False)

print("✅ Stacked all rows into uncleaned_honda_cars.csv")

✅ Stacked all rows into uncleaned_honda_cars.csv


In [92]:
# cleaning the merged csv file
# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv("uncleaned_honda_cars.csv", encoding="utf-8")

In [93]:
# -------------------------------
# 1. Clean Price Column
# -------------------------------
def clean_price(val):
    if pd.isna(val):
        return None
    val = str(val).replace("â‚¹", "").replace("₹", "").strip()
    match = re.match(r"([\d\.]+)\s*(lakh|crore)?", val.lower())
    if match:
        num = float(match.group(1))
        unit = match.group(2)
        if unit == "lakh":
            return int(num * 100000)
        elif unit == "crore":
            return int(num * 10000000)
        else:
            return int(num)
    return None

df["price"] = df["price"].apply(clean_price)

In [94]:
# -------------------------------
# 2. Clean km_driven Column
# -------------------------------
def clean_km(val):
    if pd.isna(val):
        return None
    val = re.sub(r"[^\d]", "", str(val))
    return int(val) if val else None

df["km_driven"] = df["km_driven"].apply(clean_km)


In [95]:
# -------------------------------
# 3. Clean Model Column
# -------------------------------
def clean_model(val):
    if pd.isna(val):
        return None
    return re.sub(r"\s+", " ", str(val)).strip().title()

df["model"] = df["model"].apply(clean_model)

In [96]:
# -------------------------------
# 4. Clean no_owners Column (your logic)
# -------------------------------
if "no_owners" in df.columns:
    df["no_owners"] = (
        df["no_owners"]
        .str.lower()
        .str.replace("st owner", "", regex=False)
        .str.replace("nd owner", "", regex=False)
        .str.replace("rd owner", "", regex=False)
        .str.replace("th owner", "", regex=False)
        .str.extract(r"(\d+)")  # keep only the number
        .astype(float)
    )

In [97]:
# -------------------------------
# Save cleaned dataset
# -------------------------------
df.to_csv("cleaned_honda_cars.csv", index=False, encoding="utf-8-sig")

print("✅ Final cleaning complete! Saved as 'cleaned_honda_cars.csv'")
print(df.head())


✅ Final cleaning complete! Saved as 'cleaned_honda_cars.csv'
    location  year  brand                    model   price  km_driven  \
0  hyderabad  2017  Honda    Jazz 1.2L I-Vtec V At  515000      66557   
1  hyderabad  2017  Honda    Jazz 1.2L I-Vtec V At  538000      69710   
2  hyderabad  2013  Honda                Brio V Mt  256000      47006   
3  hyderabad  2013  Honda     Amaze 1.2L I-Vtec Vx  267000      60414   
4  hyderabad  2015  Honda  City 1.5L I-Vtec Sv Cvt  463000      77391   

   no_owners transmission fuel_type  
0        1.0    Automatic    Petrol  
1        1.0    Automatic    Petrol  
2        2.0       Manual    Petrol  
3        3.0       Manual       CNG  
4        1.0    Automatic    Petrol  


In [2]:
df = pd.read_csv('cleaned_honda_cars.csv')
df.head()

,location,year,brand,model,price,km_driven,no_owners,transmission,fuel_type
0,hyderabad,2017,Honda,Jazz 1.2L I-Vtec V At,515000,66557,1.0,Automatic,Petrol
1,hyderabad,2017,Honda,Jazz 1.2L I-Vtec V At,538000,69710,1.0,Automatic,Petrol
2,hyderabad,2013,Honda,Brio V Mt,256000,47006,2.0,Manual,Petrol
3,hyderabad,2013,Honda,Amaze 1.2L I-Vtec Vx,267000,60414,3.0,Manual,CNG
4,hyderabad,2015,Honda,City 1.5L I-Vtec Sv Cvt,463000,77391,1.0,Automatic,Petrol
